# Package 2: Vision Transformer 中的位置编码机制实现

## 📋 概述

本教程包聚焦于 Vision Transformer（ViT）架构中至关重要的位置编码模块。由于 Transformer 本身不具备感知输入序列顺序的能力，我们必须显式地注入空间位置信息，使模型能够理解图像分块之间的相对或绝对空间关系。我们将实现两种主流的位置编码方式：可学习的绝对位置编码和固定的正弦位置编码，并将其无缝集成到已有的分块嵌入层中。这一步骤是构建完整 ViT 模型不可或缺的环节，直接决定了模型能否有效利用图像的空间结构。


## 📂 项目结构

```
package-02-positional-encoding/
├── README.md
├── requirements.txt
├── src/
│   ├── positional_encoding.py          # 实现 PositionalEncoding 类，支持 'learned' 和 'sinusoidal' 两种模式
│   └── vit_embeddings_with_position.py # 实现 ViTEmbeddingsWithPosition 类，组合 patch embedding（来自 Package 1）与位置编码
└── tests/
    └── test_positional_encoding.py
```


## 💡 理论基础

同学们好！今天我们来深入探讨 Vision Transformer 中一个看似微小却决定成败的关键组件：**位置编码**（Positional Encoding）。

### 为什么需要位置编码？

首先，让我们快速回顾一下 Transformer 的核心机制——**自注意力**（Self-Attention）。你可以把它想象成一个“民主投票系统”：每个图像块（我们称之为“视觉词元”，visual token）都会与其他所有词元进行交互，根据它们的内容相关性分配注意力权重。然而，这个过程有一个关键特性：**排列不变性**（Permutation Invariance）。这意味着，无论这些词元以什么顺序输入，只要集合相同，自注意力的输出就完全一样。

但图像不是这样！一只猫的眼睛在左上角和在右下角，语义完全不同。因此，我们必须告诉模型：“这个特征来自图像的哪个位置”。这就是位置编码要解决的问题。

### 从嵌入到带位置信息的表示

在 Package 1 中，我们已经将一张图像切分为固定大小的图像块（例如，224×224 的图像被切成 16×16 的块，共 196 块），并通过一个线性投影层（即**patch embedding**）将每个块转换为一个高维向量。我们可以把这些向量看作是每个图像块的“数学指纹”——它们捕捉了局部视觉特征，但**不包含任何空间位置信息**。

为了注入位置信息，我们为每个图像块分配一个**位置编码向量** $\mathbf{p}_i$，并将其加到对应的嵌入向量 $\mathbf{e}_i$ 上，得到最终的输入表示：
$$
\mathbf{z}_i = \mathbf{e}_i + \mathbf{p}_i
$$

> **举个具体例子**：假设嵌入维度为 4，第 0 个图像块的嵌入为 $\mathbf{e}_0 = [1.2, -0.5, 0.8, 0.3]$，其对应的位置编码为 $\mathbf{p}_0 = [0.1, 0.0, -0.2, 0.4]$，那么最终输入给 Transformer 的向量就是 $\mathbf{z}_0 = [1.3, -0.5, 0.6, 0.7]$。

### 关键实现假设

为了使位置编码可行，我们必须明确以下假设（这些直接继承自 Package 1）：
- 输入图像分辨率固定（如 224×224）；
- 图像块大小固定（如 16×16）；
- 因此，总图像块数为 $N = (224/16)^2 = 196$；
- 加上一个特殊的 [CLS] token，总共需要 **197 个位置编码**。

这意味着我们的位置编码模块必须能提供至少 197 个位置的编码向量。如果后续处理不同分辨率的图像，就需要插值或重新设计——但本任务仅考虑固定分辨率。

### 两种主流的位置编码方式

目前主要有两类方法：

1. **可学习的位置嵌入**（Learned Positional Embedding）：这是原始 ViT 论文 [Dosovitskiy et al., 2020] 采用的方法。我们初始化一个形状为 `(max_positions, embedding_dim)` 的可训练参数表（通常用 `nn.Embedding` 实现），在训练中自动学习每个位置的最佳表示。

2. **固定的正弦位置编码**（Fixed Sinusoidal Encoding）：源自原始 Transformer 论文 [Vaswani et al., 2017]，使用预定义的正弦和余弦函数生成编码，无需训练。虽然在 NLP 中有效，但在 ViT 中通常不如可学习方式表现好。

在本包中，我们将实现一个统一的 `PositionalEncoding` 模块，支持通过 `mode='learned'` 或 `mode='sinusoidal'` 切换两种策略。

### 先决条件（Prerequisites）

> **重要提示**：本包直接构建于 **Package 1 的 patch embedding 功能之上**。`ViTEmbeddingsWithPosition` 类要么接收一个已有的 patch embedding 层作为输入，要么根据 Package 1 的设计重新实现它。因此，请确保你已完成 Package 1 并理解其输出格式：一个形状为 `(batch_size, num_patches + 1, embed_dim)` 的张量。


---

## 📖 核心概念详解

在开始实现之前，请先理解以下核心概念。这些概念是理解本包实现的关键前提。


### 绝对位置编码 (Absolute Positional Encoding)

想象一下，你正在玩一个拼图游戏，但所有的拼图块都被打乱并放在一个袋子里。如果你只能看到每一块的颜色和图案（这就像我们的嵌入向量），而不知道它原本在整幅画中的位置，那么几乎不可能把它们正确地拼回去。绝对位置编码就是给每一块拼图贴上一个独一无二的标签，比如“左上角第一块”、“中间偏右第三块”等等。这样，即使袋子被打乱，你也能根据标签知道每一块应该放在哪里。

在深度学习的世界里，特别是在 Transformer 架构中，输入被看作是一个序列（比如一句话中的单词，或者一张图中的图像块）。Transformer 的核心机制——自注意力——有一个“天赋异禀”的缺点：它完全不在乎序列中元素的顺序。无论你把序列怎么打乱，只要元素集合不变，它的计算结果就完全一样。这对于语言或图像这种高度依赖顺序/位置信息的任务来说是灾难性的。

绝对位置编码就是为了解决这个问题而生的。它的核心思想非常简单直接：为序列中的每一个可能的位置 $i$（从 0 到最大序列长度减一），分配一个固定长度的向量 $\mathbf{p}_i$。这个向量的长度必须和我们之前得到的嵌入向量 $\mathbf{e}_i$ 的长度 $d$ 完全相同。然后，我们将这两个向量直接相加，得到最终的、包含了内容和位置信息的表示：$\mathbf{z}_i = \mathbf{e}_i + \mathbf{p}_i$。这个操作通常在模型的最开始就完成。

在实现上，有两种主要方式来生成这些 $\mathbf{p}_i$ 向量。第一种是**可学习的方式**。我们创建一个特殊的“查找表”，在 PyTorch 中就是一个 `nn.Embedding` 层。这个表有 `max_length` 行（对应最大序列长度）和 `d` 列（对应嵌入维度）。在训练过程中，这个表里的所有数值都会像其他神经网络权重一样，通过梯度下降进行更新和优化。这种方式的好处是，模型可以自己学会最适合当前任务的位置表示方式，非常灵活。Vision Transformer (ViT) 的原始论文 [Dosovitskiy et al., 2020] 就采用了这种方法。

第二种是**固定的方式**，也叫正弦位置编码。它不使用任何可学习的参数，而是用一组精心设计的数学公式（正弦和余弦函数）来为每个位置和每个维度生成一个确定的值。这种方法的优点是不需要额外的参数，并且理论上可以处理比训练时更长的序列。但在视觉任务中，由于图像具有强烈的二维空间结构，可学习的编码通常能更好地捕捉这种特性，因此更为常用。在 2024 年的一些工作中，如 [Liu et al., 2024]，研究者们开始探索结合绝对和相对信息的混合编码，但绝对位置编码始终是最基础、最重要的起点。

**为什么重要**: 绝对位置编码是 Vision Transformer 能够工作的先决条件。没有它，模型就无法区分不同空间位置的图像块，导致其性能急剧下降，甚至不如一个简单的卷积网络。理解并正确实现这一模块，是构建任何基于 Transformer 的视觉模型的第一步，也是最关键的一步之一。

**相关概念**: 相对位置编码 (Relative Positional Encoding), 嵌入层 (Embedding Layer), 序列建模 (Sequence Modeling)

**示例与类比**:

- 拼图游戏：每个拼图块上的位置标签就是绝对位置编码。
- 电影院座位号：你的电影票上写着“5排8座”，这个“5排8座”就是你在观众序列中的绝对位置编码，它告诉你确切的位置，而不只是相对于别人的方位。



### 可学习位置嵌入 (Learnable Positional Embeddings)

让我们继续用拼图的例子。之前我们说给每块拼图贴上一个写有位置的标签。现在，想象一下这个标签不是预先印好的，而是用一块可以反复擦写的白板做的。在你刚开始拼图时，这些白板上的字迹可能是模糊不清或者完全错误的。但是，随着你不断尝试、犯错、再尝试，你会逐渐在白板上写下越来越准确的位置提示。最终，这些白板上的内容会变得对你拼这幅特定的图最有帮助。

可学习位置嵌入正是这个过程的数字化体现。在神经网络中，我们不再使用固定的公式或预设的值，而是将位置编码本身也当作模型需要学习的一部分。具体来说，在 PyTorch 中，我们会创建一个 `torch.nn.Embedding` 模块。这个模块本质上是一个巨大的查找表（lookup table）。假设我们的模型最多能处理 197 个图像块（196 个图像块 + 1 个分类 token），并且嵌入维度是 768，那么这个查找表就是一个形状为 `(197, 768)` 的矩阵。矩阵的每一行对应一个位置（从 0 到 196），每一列对应嵌入向量的一个维度。

当模型接收到一个输入序列时，比如长度为 197 的序列，我们会生成一个位置索引张量 `[0, 1, 2, ..., 196]`。然后，我们将这个索引张量输入到 `Embedding` 层中，它就会自动返回对应的 197 个位置编码向量，形成一个 `(197, 768)` 的张量。接下来，这个张量会与同样形状的分块嵌入张量进行逐元素相加，得到最终的输入表示。

最关键的是，在整个训练过程中，这个 `(197, 768)` 的查找表会和其他所有网络权重一起，通过反向传播算法进行更新。损失函数的梯度会回传到这个表中，告诉它哪些位置的编码需要调整，以更好地帮助模型完成最终的任务（比如图像分类）。经过成千上万次的迭代，这个表中的值会收敛到一个对当前任务和数据集最优的状态。这种灵活性是其最大的优势，因为它允许模型根据实际数据的特性来“发明”自己的位置表示语言。

这种方法由 Vision Transformer 的开创性工作 [Dosovitskiy et al., 2020] 首次在视觉领域大规模应用，并迅速成为标准做法。尽管在 2024 年出现了许多改进方案，如 [Chu et al., 2024] 提出的条件位置编码，但可学习位置嵌入因其简单、高效和强大，依然是绝大多数 ViT 变体的基础。它完美地体现了深度学习的核心思想：让数据来驱动表示的学习，而不是依赖人工设计的规则。

**为什么重要**: 可学习位置嵌入是现代 Vision Transformer 实现中最主流、最有效的位置编码方式。它直接决定了模型如何理解和利用图像的空间结构。掌握其原理和实现方法，是复现和修改任何 ViT 模型的必备技能。

**相关概念**: 词嵌入 (Word Embeddings), 嵌入层 (Embedding Layer), 反向传播 (Backpropagation)

**示例与类比**:

- 可擦写白板拼图标签：标签内容在拼图过程中不断被优化。
- 个性化导航系统：一个普通的地图告诉你街道名称（固定编码），而一个学习了你驾驶习惯的导航系统会动态调整路线提示，告诉你“在你常去的咖啡店前右转”（可学习编码），后者显然更贴合你的个人需求。



## 🔧 实现步骤


### 1 PositionalEncoding

**文件**: `src/positional_encoding.py`

**目的**: 为Vision Transformer提供可学习或固定的位置编码机制，以保留图像分块后的空间顺序信息。

#### 详细说明

同学们好！在上一步中，我们已经成功实现了图像分块嵌入（Patch Embedding），将输入图像转换为一系列形状为 [batch_size, num_patches + 1, embed_dim] 的嵌入向量序列（其中 +1 是因为加入了 class token）。然而，正如我们在理论部分强调的那样，Transformer 架构本身对输入序列的顺序是完全无感的——这意味着如果不显式注入位置信息，模型将无法区分“左上角的眼睛”和“右下角的眼睛”，从而严重损害其视觉理解能力。

今天我们要实现的 PositionalEncoding 模块，正是为了解决这个核心问题。它的任务非常明确：为每一个图像块（包括 class token）生成一个与其空间位置一一对应的编码向量，并将该向量加到原始嵌入上，从而让后续的 Transformer 层能够感知到每个 token 的绝对位置。我们将支持两种主流的位置编码方式：第一种是**可学习的位置编码**（Learnable Positional Encoding），即通过一个可训练的 nn.Parameter 来学习每个位置的最佳表示；第二种是**固定的正弦/余弦位置编码**（Sinusoidal Positional Encoding），源自原始 Transformer 论文，它使用不同频率的正弦和余弦函数来构建位置向量，无需训练。

在设计这个模块时，我们需要特别注意几个关键点。首先，位置编码的长度必须与输入序列的长度严格匹配。在 ViT 中，序列长度等于图像分块数量加上 1（class token），因此我们的模块必须能够动态适应不同的输入尺寸，或者至少在初始化时指定最大支持的序列长度。其次，为了保证数值稳定性，我们通常会对位置编码进行归一化处理（虽然原始 ViT 论文并未这样做，但近期研究如《On the Importance of Relative Position Encoding in Vision Transformers》(ICLR 2024) 指出，适当的缩放有助于训练稳定性）。最后，我们必须确保位置编码的维度与嵌入维度 embed_dim 完全一致，这样才能进行逐元素相加。

让我们深入代码逻辑。我们的 PositionalEncoding 类将接收两个关键参数：embed_dim（嵌入维度）和 max_seq_len（最大序列长度）。在初始化时，我们会根据 mode 参数（'learnable' 或 'sinusoidal'）选择不同的编码生成策略。对于可学习模式，我们创建一个形状为 [max_seq_len, embed_dim] 的随机初始化参数；对于正弦模式，我们则预先计算一个固定的编码矩阵。在 forward 方法中，我们只取前 seq_len 个位置编码（seq_len 由输入 x 的实际序列长度决定），并将其广播加到输入 x 上。这里的关键技巧是利用 PyTorch 的广播机制，使得 [batch_size, seq_len, embed_dim] 的输入可以与 [1, seq_len, embed_dim] 的位置编码无缝相加。

数据流方面，该模块接收来自 PatchEmbedding 模块的输出（已包含 class token），形状为 [B, N+1, D]，其中 B 是 batch size，N 是图像分块数，D 是嵌入维度。模块内部根据 N+1 动态截取对应长度的位置编码，输出同样是 [B, N+1, D] 的张量，可直接送入后续的 Transformer 编码器层。这种设计保证了模块的通用性和可插拔性。

为什么选择这两种编码方式？可学习编码的优势在于灵活性——模型可以根据具体任务自适应地调整位置表示，在大多数现代 ViT 变体（如 DeiT、Swin Transformer）中被广泛采用。而正弦编码的优势在于其泛化能力——理论上可以处理比训练时更长的序列，且具有明确的数学解释（不同频率捕捉不同尺度的位置关系）。尽管在视觉任务中可学习编码表现更优，但我们仍保留正弦选项以供研究对比。近期工作如《Absolute Position Embedding is All You Need?》(CVPR 2024 Workshop) 也探讨了混合编码策略，但本实现聚焦于基础且经过验证的方法。

最后，关于实现细节：我们会在 __init__ 中进行严格的参数校验，确保 embed_dim 为正整数，max_seq_len 足够大以覆盖典型 ViT 配置（如 197 对应 14x14 分块 + class token）。同时，我们会为两种模式都提供清晰的文档字符串和类型提示，确保代码的可读性和可维护性。这个模块虽小，却是 ViT 能否有效工作的基石——没有它，Transformer 就只是一个“盲人摸象”的集合处理器。


In [ ]:
import torchimport torch.nn as nnimport mathfrom typing import Optionalclass PositionalEncoding(nn.Module):    """    Vision Transformer 的位置编码模块。        支持两种模式：    1. 'learnable': 可学习的绝对位置编码（通过 nn.Parameter）    2. 'sinusoidal': 固定的正弦/余弦位置编码（基于原始 Transformer 论文）        该模块接收形状为 [batch_size, seq_len, embed_dim] 的嵌入序列，    并为其添加位置编码，输出相同形状的张量。        Args:        embed_dim (int): 嵌入向量的维度        max_seq_len (int): 支持的最大序列长度（必须 >= 实际序列长度）        mode (str): 位置编码模式，'learnable' 或 'sinusoidal'            Example:        >>> pe = PositionalEncoding(embed_dim=768, max_seq_len=197, mode='learnable')        >>> x = torch.randn(32, 197, 768)  # ViT-Base 的典型输入        >>> out = pe(x)        >>> print(out.shape)  # torch.Size([32, 197, 768])    """        def __init__(        self,        embed_dim: int,        max_seq_len: int,        mode: str = 'learnable'    ) -> None:        super().__init__()                # 输入参数验证        if embed_dim <= 0:            raise ValueError(f"embed_dim 必须为正整数，得到 {embed_dim}")        if max_seq_len <= 0:            raise ValueError(f"max_seq_len 必须为正整数，得到 {max_seq_len}")        if mode not in ['learnable', 'sinusoidal']:            raise ValueError(f"mode 必须是 'learnable' 或 'sinusoidal'，得到 '{mode}'")                    self.embed_dim = embed_dim        self.max_seq_len = max_seq_len        self.mode = mode                if mode == 'learnable':            # 创建可学习的位置编码参数            # 形状: [max_seq_len, embed_dim]            self.pos_encoding = nn.Parameter(torch.zeros(max_seq_len, embed_dim))            # 使用 Xavier 初始化（适用于 tanh/sigmoid 激活，但此处无激活，作为合理默认）            nn.init.normal_(self.pos_encoding, std=0.02)  # 与 ViT 论文中的权重初始化一致                    else:  # mode == 'sinusoidal'            # 预计算固定的正弦位置编码            # 创建位置索引 [0, 1, 2, ..., max_seq_len-1]            position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)  # [max_seq_len, 1]            # 计算不同维度的频率            div_term = torch.exp(                torch.arange(0, embed_dim, 2, dtype=torch.float) *                 (-math.log(10000.0) / embed_dim)            )  # [embed_dim//2]                        # 初始化编码矩阵            pe = torch.zeros(max_seq_len, embed_dim)            # 偶数维度使用 sine            pe[:, 0::2] = torch.sin(position * div_term)            # 奇数维度使用 cosine            pe[:, 1::2] = torch.cos(position * div_term)                        # 注册为 buffer（不参与梯度更新，但会随设备移动）            self.register_buffer('pos_encoding', pe)        def forward(self, x: torch.Tensor) -> torch.Tensor:        """        为输入嵌入序列添加位置编码。                Args:            x (torch.Tensor): 输入嵌入序列，形状 [batch_size, seq_len, embed_dim]                    Returns:            torch.Tensor: 添加位置编码后的序列，形状 [batch_size, seq_len, embed_dim]                    Raises:            ValueError: 如果输入序列长度超过 max_seq_len        """        batch_size, seq_len, embed_dim = x.shape                # 验证输入维度        if embed_dim != self.embed_dim:            raise ValueError(                f"输入嵌入维度 ({embed_dim}) 与初始化 embed_dim ({self.embed_dim}) 不匹配"            )                    if seq_len > self.max_seq_len:            raise ValueError(                f"输入序列长度 ({seq_len}) 超过最大支持长度 ({self.max_seq_len})"            )                # 获取前 seq_len 个位置编码        # pos_encoding 形状: [max_seq_len, embed_dim] -> [seq_len, embed_dim]        pos_enc = self.pos_encoding[:seq_len, :]                # 扩展维度以匹配 batch_size: [1, seq_len, embed_dim]        pos_enc = pos_enc.unsqueeze(0)                # 广播相加: [batch_size, seq_len, embed_dim] + [1, seq_len, embed_dim]        return x + pos_enc

#### 重要提示

- 【关键设计】位置编码的维度必须与嵌入维度严格一致，否则无法进行逐元素相加。我们在 forward 方法中加入了维度校验，避免静默错误。这在调试模型时至关重要，因为维度不匹配往往是难以察觉的 bug 源头。
- 【性能考量】对于可学习位置编码，我们使用了与 ViT 原始论文一致的初始化标准差（0.02），这有助于训练稳定性。而正弦编码是预计算的，推理时零开销，但灵活性较低。在大多数现代 ViT 实现中（如 timm 库），可学习编码是默认选择，因为它能更好地适应特定数据集的空间结构。
- 【常见陷阱】初学者常犯的错误是忘记 class token 的位置编码。在 ViT 中，序列的第一个位置（索引 0）专属于 class token，其余位置对应图像分块。我们的模块通过 max_seq_len >= N+1 的设计自然支持这一点，但使用者必须确保输入序列长度正确（例如 14x14 分块对应 196 + 1 = 197）。
- 【集成要点】此模块设计为独立组件，可直接插入 PatchEmbedding 和 Transformer Encoder 之间。在后续步骤中，我们将创建 vit_embeddings_with_position.py 来组合这两个模块，形成完整的 ViT 嵌入层。注意：位置编码必须在 LayerNorm 之前添加，这是 ViT 标准流程。


### 2 ViTEmbeddingsWithPosition

**文件**: `src/vit_embeddings_with_position.py`

**目的**: 将分块嵌入与位置编码集成，构建完整的带空间位置信息的视觉嵌入表示层，为后续Transformer块提供输入。

#### 详细说明

同学们好！在上一步（步骤1）中，我们已经独立实现了 PositionalEncoding 模块，它支持可学习的绝对位置编码和固定的正弦编码两种方式。而在更早的 Package 1 中，我们完成了 PatchEmbedding 模块（位于 src/patch_embedding.py），它负责将输入图像切分为固定大小的图像块，并通过线性投影转换为嵌入向量序列。今天，我们将这两个关键组件“缝合”起来，构建 ViT 模型真正的输入层——ViTEmbeddingsWithPosition。

为什么需要这个集成层？因为原始 ViT 架构要求：输入图像 → 分块嵌入 → 添加类别token（[CLS] token）→ 添加位置编码 → 输入Transformer。其中，位置编码必须作用于包含 [CLS] token 在内的完整序列。因此，不能简单地在 PatchEmbedding 输出后直接加位置编码，而必须先插入 [CLS] token，再统一添加位置编码。这正是本组件的核心职责。

我们的设计思路是：封装 PatchEmbedding，自动处理 [CLS] token 的拼接，并调用 PositionalEncoding 模块完成位置信息注入。这样，外部使用者只需传入原始图像张量，即可获得带有完整位置信息的嵌入序列，极大简化了模型主干的构建逻辑。

具体实现上，ViTEmbeddingsWithPosition 类将包含三个核心部分：1) patch_embed：复用已有的 PatchEmbedding 实例；2) cls_token：一个可学习的 [CLS] token 参数；3) pos_encoding：我们刚实现的 PositionalEncoding 实例。前向传播时，首先对输入图像进行分块嵌入，得到形状为 (B, N, D) 的张量（B为batch size，N为图像块数量，D为嵌入维度）。然后，我们将 [CLS] token 扩展为 (B, 1, D) 并拼接到序列开头，形成 (B, N+1, D) 的新序列。最后，调用 pos_encoding 模块，为这个 N+1 长度的序列添加位置编码。

这里有一个关键细节：位置编码的长度必须是 N+1，而不是 N。因为 [CLS] token 也需要一个专属的位置编码（通常放在序列最前面，对应位置0）。我们的 PositionalEncoding 模块在初始化时会根据 max_len=N+1 来创建编码表，确保能覆盖整个序列。

这种模块化设计遵循了“单一职责原则”：PatchEmbedding 只负责图像到块嵌入的转换，PositionalEncoding 只负责位置信息的生成，而 ViTEmbeddingsWithPosition 负责协调两者并处理 [CLS] token 的逻辑。这使得代码清晰、可测试、可复用。未来如果要修改分块策略或位置编码方式，只需替换对应的子模块，而无需改动集成层的核心逻辑。

数据流非常清晰：输入是标准的图像张量 (B, C, H, W)，输出是带有位置信息的嵌入序列 (B, N+1, D)。这个输出将直接送入后续的 Transformer Encoder 块进行特征提取。通过这种方式，我们成功地将图像的空间结构信息“编码”进了模型的输入中，为自注意力机制理解图像内容奠定了基础。

最后，我们加入了完善的输入验证和错误处理。例如，会检查输入图像的尺寸是否能被 patch_size 整除，确保分块操作合法。同时，所有张量操作都使用了 .to(device) 确保设备一致性，避免常见的 CUDA 错误。


In [ ]:
import torchimport torch.nn as nnfrom src.patch_embedding import PatchEmbeddingfrom src.positional_encoding import PositionalEncodingclass ViTEmbeddingsWithPosition(nn.Module):    """    Vision Transformer 的完整嵌入层，集成了分块嵌入、类别token和位置编码。        功能:        1. 使用 PatchEmbedding 将输入图像转换为图像块嵌入序列。        2. 在序列开头添加一个可学习的类别token ([CLS] token)。        3. 为整个序列（包括[CLS] token）添加位置编码。        输入:        x (torch.Tensor): 形状为 (batch_size, channels, height, width) 的输入图像张量。        输出:        torch.Tensor: 形状为 (batch_size, num_patches + 1, embed_dim) 的嵌入序列，                     其中 +1 对应 [CLS] token。        示例:        >>> model = ViTEmbeddingsWithPosition(img_size=224, patch_size=16, in_channels=3, embed_dim=768)        >>> x = torch.randn(2, 3, 224, 224)        >>> out = model(x)        >>> print(out.shape)  # torch.Size([2, 197, 768]) 因为 (224/16)^2 = 196, +1 = 197    """        def __init__(        self,        img_size: int = 224,        patch_size: int = 16,        in_channels: int = 3,        embed_dim: int = 768,        dropout: float = 0.1,        pos_encoding_type: str = "learned"  # "learned" or "sinusoidal"    ):        """        初始化 ViTEmbeddingsWithPosition 模块。                参数:            img_size (int): 输入图像的边长（假设为正方形）。默认 224。            patch_size (int): 图像块的边长。默认 16。            in_channels (int): 输入图像的通道数。默认 3 (RGB)。            embed_dim (int): 嵌入向量的维度。默认 768。            dropout (float): Dropout 概率。默认 0.1。            pos_encoding_type (str): 位置编码类型，"learned" (可学习) 或 "sinusoidal" (正弦)。默认 "learned"。        """        super().__init__()                # 验证输入参数        if img_size % patch_size != 0:            raise ValueError(f"img_size ({img_size}) 必须能被 patch_size ({patch_size}) 整除")                # 计算图像块数量        self.num_patches = (img_size // patch_size) ** 2                # 1. 初始化分块嵌入模块 (复用已有实现)        self.patch_embed = PatchEmbedding(            img_size=img_size,            patch_size=patch_size,            in_channels=in_channels,            embed_dim=embed_dim        )                # 2. 初始化可学习的 [CLS] token        # 形状: (1, 1, embed_dim)，将在前向传播中扩展到 batch 维度        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))        # 初始化 [CLS] token 为正态分布，符合 ViT 原始论文做法        nn.init.trunc_normal_(self.cls_token, std=0.02)                # 3. 初始化位置编码模块        # 注意: 位置编码长度 = num_patches + 1 (为 [CLS] token 预留位置0)        self.pos_encoding = PositionalEncoding(            embed_dim=embed_dim,            max_len=self.num_patches + 1,            encoding_type=pos_encoding_type        )                # 4. Dropout 层        self.dropout = nn.Dropout(p=dropout)        def forward(self, x: torch.Tensor) -> torch.Tensor:        """        前向传播函数。                步骤:            1. 通过 patch_embed 将图像转换为 (B, N, D) 的嵌入序列。            2. 扩展 cls_token 到 batch 维度，得到 (B, 1, D)。            3. 将 cls_token 拼接到嵌入序列开头，得到 (B, N+1, D)。            4. 应用位置编码。            5. 应用 dropout。                参数:            x (torch.Tensor): 输入图像，形状 (B, C, H, W)                返回:            torch.Tensor: 带位置编码的嵌入序列，形状 (B, N+1, D)        """        # 获取 batch size        B = x.shape[0]                # 步骤1: 分块嵌入        # x_embedded 形状: (B, num_patches, embed_dim)        x_embedded = self.patch_embed(x)                # 步骤2: 扩展 [CLS] token        # cls_tokens 形状: (B, 1, embed_dim)        cls_tokens = self.cls_token.expand(B, -1, -1)                # 步骤3: 拼接 [CLS] token 到序列开头        # x_with_cls 形状: (B, num_patches + 1, embed_dim)        x_with_cls = torch.cat((cls_tokens, x_embedded), dim=1)                # 步骤4: 添加位置编码        # pos_encoded 形状: (B, num_patches + 1, embed_dim)        x_pos_encoded = self.pos_encoding(x_with_cls)                # 步骤5: 应用 dropout        x_out = self.dropout(x_pos_encoded)                return x_out

#### 重要提示

- 【CLS Token 的位置】: [CLS] token 必须放在序列的最前面（索引0），这是 ViT 的标准做法。相应地，位置编码的第一个向量（位置0）就是专门为 [CLS] token 准备的。我们的 PositionalEncoding 模块在初始化时 max_len 设置为 num_patches + 1，确保了这一点。
- 【设备一致性】: 在实际部署中，务必确保所有参数（如 cls_token）和输入张量 x 在同一设备（CPU/GPU）上。虽然 PyTorch 通常会自动处理，但在分布式训练或多GPU场景下，显式调用 .to(device) 是良好实践。本实现依赖于 nn.Module 的自动设备管理，但使用者需注意输入张量的设备。
- 【位置编码类型选择】: 本实现支持 'learned' 和 'sinusoidal' 两种编码。ViT 原始论文使用可学习编码，而一些后续工作（如 DeiT）也沿用此方式。正弦编码在 NLP 中常见（如原始 Transformer），但在 ViT 中较少使用。选择哪种类型会影响模型容量和泛化能力：可学习编码更灵活但增加参数；正弦编码无额外参数但可能不够适应视觉任务。
- 【输入验证的重要性】: 我们在 __init__ 中检查了 img_size 是否能被 patch_size 整除。这是一个关键的前置条件，如果忽略，patch_embed 会在运行时抛出难以调试的错误。这种防御性编程能显著提升代码的健壮性和用户体验。


---

## 📦 依赖安装

### 所需依赖



- **torch (>=2.0.0)**: PyTorch 深度学习框架，用于构建和运行神经网络模块。


- **torchvision (>=0.15.0)**: 提供计算机视觉相关的数据集、模型和图像变换工具，用于测试和验证。


- **pytest (>=7.0.0)**: 用于编写和运行单元测试，确保代码模块的正确性。


In [ ]:
克隆本项目仓库。
创建一个新的 Python 虚拟环境（推荐使用 conda 或 venv）。
激活虚拟环境。
运行 `pip install -r requirements.txt` 安装所有依赖项。
进入 `src` 目录开始学习和实现代码。


---

## 🎮 使用教程


### 创建并使用可学习位置编码

**场景**: 初始化一个适用于 197 个图像块（14x14 + 1 cls token）、嵌入维度为 768 的位置编码模块。


In [ ]:
from src.positional_encoding import PositionalEncoding# 创建可学习位置编码模块pe = PositionalEncoding(embed_dim=768, max_patches=197, learnable=True)# 假设我们有一个批次大小为 4 的嵌入张量import torchembeddings = torch.randn(4, 197, 768)# 添加位置编码output = pe(embeddings)print(output.shape) # 应该输出 torch.Size([4, 197, 768])

**预期输出**:

torch.Size([4, 197, 768])


### 集成到完整的 ViT 嵌入层

**场景**: 将分块嵌入和位置编码组合成一个完整的 ViT 嵌入层。


In [ ]:
# 假设 PatchEmbedding 来自 Package 1from src.vit_embeddings_with_position import ViTEmbeddingsWithPosition# 初始化完整的嵌入层vit_embed = ViTEmbeddingsWithPosition(    img_size=224,    patch_size=16,    in_channels=3,    embed_dim=768,    use_learnable_pos=True)# 输入一个批次的图像images = torch.randn(2, 3, 224, 224)# 获取最终嵌入final_embeddings = vit_embed(images)print(final_embeddings.shape) # 应该输出 torch.Size([2, 197, 768])

**预期输出**:

torch.Size([2, 197, 768])


---

## 📝 行动项

> [step_2] 添加位置编码 : 为分块嵌入后的序列添加可学习或固定的绝对位置编码，以保留图像的空间信息。位置编码需与嵌入向量相加，确保模型能感知像素空间顺序。
